In [ ]:
"""
Next Word Predictor using PyTorch (LSTM Language Model)
========================================================
A complete implementation including:
  - Data preprocessing & vocabulary building
  - LSTM-based language model
  - Training loop with loss tracking
  - Top-k next word prediction with temperature sampling
  - Model save/load
  - Interactive prediction demo
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import re
import os
from collections import Counter


# ─────────────────────────────────────────────
# 1. CONFIGURATION
# ─────────────────────────────────────────────

class Config:
    # Model hyperparameters
    EMBED_DIM    = 128       # Word embedding size
    HIDDEN_DIM   = 256       # LSTM hidden state size
    NUM_LAYERS   = 2         # Number of LSTM layers
    DROPOUT      = 0.3       # Dropout probability

    # Training settings
    SEQ_LEN      = 5         # Context window (n-gram size)
    BATCH_SIZE   = 64
    EPOCHS       = 30
    LEARNING_RATE = 0.001
    CLIP_GRAD    = 5.0       # Gradient clipping

    # Vocabulary
    MIN_FREQ     = 2         # Minimum word frequency to include
    VOCAB_SIZE   = None      # Set automatically after building vocab

    # Paths
    MODEL_PATH   = "next_word_model.pt"

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ─────────────────────────────────────────────
# 2. SAMPLE TRAINING CORPUS
#    Replace with your own text for better results
# ─────────────────────────────────────────────

SAMPLE_TEXT = """
The quick brown fox jumps over the lazy dog.
Machine learning is a subset of artificial intelligence.
Deep learning uses neural networks with many layers.
Natural language processing helps computers understand human language.
Python is a popular programming language for data science.
PyTorch is an open source machine learning framework.
The cat sat on the mat near the door.
She sells seashells by the seashore every morning.
To be or not to be that is the question asked by Shakespeare.
All that glitters is not gold said the wise man.
A journey of a thousand miles begins with a single step.
The early bird catches the worm in the morning light.
Actions speak louder than words in every situation.
Better late than never to start learning something new.
Knowledge is power when applied with wisdom and care.
The future belongs to those who believe in their dreams.
In the beginning there was nothing and then there was something.
Once upon a time in a land far away there lived a king.
The sun rises in the east and sets in the west every day.
Water flows downhill following the path of least resistance.
Science is the pursuit of knowledge through observation and experiment.
Music is the language of the soul that transcends barriers.
Reading books opens the mind to new worlds and ideas.
The human brain is the most complex organ in the body.
Artificial intelligence will transform every industry in the coming years.
Data is the new oil that powers the digital economy.
Computers have revolutionized the way we live and work.
The internet connects billions of people around the world.
Software engineering requires both creativity and logical thinking.
Open source projects are built by communities of developers.
"""


# ─────────────────────────────────────────────
# 3. TOKENIZER & VOCABULARY
# ─────────────────────────────────────────────

class Vocabulary:
    PAD_TOKEN = "<PAD>"
    UNK_TOKEN = "<UNK>"

    def __init__(self, min_freq: int = 2):
        self.min_freq = min_freq
        self.word2idx = {}
        self.idx2word = {}
        self.word_freq = Counter()

    def build(self, tokens: list[str]):
        """Build vocabulary from a list of tokens."""
        self.word_freq.update(tokens)

        # Reserve special tokens at index 0 and 1
        special = [self.PAD_TOKEN, self.UNK_TOKEN]
        filtered = [w for w, c in self.word_freq.items() if c >= self.min_freq]

        all_words = special + sorted(filtered)
        self.word2idx = {w: i for i, w in enumerate(all_words)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}

        print(f"  Vocabulary size : {len(self.word2idx)}")
        print(f"  Total tokens    : {len(tokens)}")

    def encode(self, word: str) -> int:
        return self.word2idx.get(word, self.word2idx[self.UNK_TOKEN])

    def decode(self, idx: int) -> str:
        return self.idx2word.get(idx, self.UNK_TOKEN)

    def __len__(self):
        return len(self.word2idx)


def tokenize(text: str) -> list[str]:
    """Lowercase and split on non-alphanumeric characters."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()


# ─────────────────────────────────────────────
# 4. DATASET
# ─────────────────────────────────────────────

class NGramDataset(Dataset):
    """
    Each sample: (context_ids, target_id)
    context_ids  : tensor of shape (SEQ_LEN,)
    target_id    : scalar tensor (next word index)
    """

    def __init__(self, tokens: list[str], vocab: Vocabulary, seq_len: int):
        self.seq_len = seq_len
        self.vocab   = vocab
        self.data    = []

        ids = [vocab.encode(t) for t in tokens]

        for i in range(len(ids) - seq_len):
            context = ids[i : i + seq_len]
            target  = ids[i + seq_len]
            self.data.append((context, target))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        context, target = self.data[idx]
        return (
            torch.tensor(context, dtype=torch.long),
            torch.tensor(target,  dtype=torch.long),
        )


# ─────────────────────────────────────────────
# 5. LSTM LANGUAGE MODEL
# ─────────────────────────────────────────────

class NextWordLSTM(nn.Module):
    """
    Architecture:
      Embedding → LSTM (stacked) → Dropout → Linear → LogSoftmax
    """

    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, num_layers: int, dropout: float):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0,      # PAD_TOKEN is always at index 0
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        """
        x      : (batch, seq_len)
        returns: logits (batch, vocab_size), hidden state
        """
        emb = self.dropout(self.embedding(x))       # (B, T, E)
        out, hidden = self.lstm(emb, hidden)         # (B, T, H)
        out = self.dropout(out[:, -1, :])            # last timestep (B, H)
        logits = self.fc(out)                        # (B, V)
        return logits, hidden


# ─────────────────────────────────────────────
# 6. TRAINING
# ─────────────────────────────────────────────

def train_epoch(model, loader, optimizer, criterion, device, clip):
    model.train()
    total_loss = 0.0

    for contexts, targets in loader:
        contexts = contexts.to(device)
        targets  = targets.to(device)

        optimizer.zero_grad()
        logits, _ = model(contexts)
        loss = criterion(logits, targets)
        loss.backward()

        # Gradient clipping prevents exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for contexts, targets in loader:
            contexts = contexts.to(device)
            targets  = targets.to(device)
            logits, _ = model(contexts)
            loss = criterion(logits, targets)
            total_loss += loss.item()

    return total_loss / len(loader)


def train(model, train_loader, val_loader, config):
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore PAD
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=3, factor=0.5, verbose=True
    )

    best_val_loss = float("inf")

    print(f"\n{'Epoch':>6}  {'Train Loss':>12}  {'Val Loss':>10}  {'Perplexity':>12}")
    print("─" * 48)

    for epoch in range(1, config.EPOCHS + 1):
        tr_loss  = train_epoch(model, train_loader, optimizer,
                               criterion, config.DEVICE, config.CLIP_GRAD)
        val_loss = evaluate(model, val_loader, criterion, config.DEVICE)
        ppl      = np.exp(val_loss)

        scheduler.step(val_loss)

        print(f"{epoch:>6}  {tr_loss:>12.4f}  {val_loss:>10.4f}  {ppl:>12.2f}")

        # Save best checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "epoch"      : epoch,
                "model_state": model.state_dict(),
                "config"     : config,
            }, config.MODEL_PATH)

    print(f"\nBest model saved → {config.MODEL_PATH}  (val_loss={best_val_loss:.4f})")


# ─────────────────────────────────────────────
# 7. PREDICTION
# ─────────────────────────────────────────────

def predict_next_words(
    model: NextWordLSTM,
    vocab: Vocabulary,
    seed_text: str,
    seq_len: int,
    top_k: int = 5,
    temperature: float = 1.0,
    device: torch.device = torch.device("cpu"),
) -> list[tuple[str, float]]:
    """
    Predict the top-k next words given a seed phrase.

    temperature < 1  → sharper / more confident distribution
    temperature > 1  → flatter / more random distribution
    """
    model.eval()

    tokens = tokenize(seed_text)
    if len(tokens) < seq_len:
        # Pad on the left if seed is shorter than context window
        tokens = [Vocabulary.PAD_TOKEN] * (seq_len - len(tokens)) + tokens
    tokens = tokens[-seq_len:]  # take last seq_len tokens

    ids = torch.tensor(
        [vocab.encode(t) for t in tokens],
        dtype=torch.long,
    ).unsqueeze(0).to(device)  # (1, seq_len)

    with torch.no_grad():
        logits, _ = model(ids)                        # (1, V)
        logits    = logits.squeeze(0) / temperature   # (V,)
        probs     = torch.softmax(logits, dim=-1)

    top_probs, top_ids = torch.topk(probs, k=top_k)

    results = []
    for prob, idx in zip(top_probs.cpu().tolist(), top_ids.cpu().tolist()):
        word = vocab.decode(idx)
        if word not in (Vocabulary.PAD_TOKEN, Vocabulary.UNK_TOKEN):
            results.append((word, round(prob, 4)))

    return results


def generate_sentence(
    model: NextWordLSTM,
    vocab: Vocabulary,
    seed_text: str,
    seq_len: int,
    num_words: int = 10,
    temperature: float = 0.8,
    device: torch.device = torch.device("cpu"),
) -> str:
    """Greedily generate `num_words` words after the seed."""
    model.eval()
    tokens = tokenize(seed_text)

    for _ in range(num_words):
        context = tokens[-seq_len:] if len(tokens) >= seq_len \
                  else [Vocabulary.PAD_TOKEN] * (seq_len - len(tokens)) + tokens

        ids = torch.tensor(
            [vocab.encode(t) for t in context],
            dtype=torch.long,
        ).unsqueeze(0).to(device)

        with torch.no_grad():
            logits, _ = model(ids)
            logits    = logits.squeeze(0) / temperature
            probs     = torch.softmax(logits, dim=-1)

        # Sample from distribution (not pure greedy) for natural output
        next_id   = torch.multinomial(probs, num_samples=1).item()
        next_word = vocab.decode(next_id)

        if next_word in (Vocabulary.PAD_TOKEN, Vocabulary.UNK_TOKEN):
            next_word = "<unk>"

        tokens.append(next_word)

    return " ".join(tokens)


# ─────────────────────────────────────────────
# 8. MAIN — PIPELINE
# ─────────────────────────────────────────────

def main():
    config = Config()
    print(f"Device: {config.DEVICE}")

    # ── Preprocessing ──────────────────────────
    print("\n[1/4] Preprocessing corpus …")
    tokens = tokenize(SAMPLE_TEXT)

    vocab = Vocabulary(min_freq=config.MIN_FREQ)
    vocab.build(tokens)
    config.VOCAB_SIZE = len(vocab)

    # ── Dataset & DataLoaders ──────────────────
    print("\n[2/4] Building dataset …")
    dataset = NGramDataset(tokens, vocab, seq_len=config.SEQ_LEN)

    val_size   = max(1, int(0.1 * len(dataset)))
    train_size = len(dataset) - val_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=config.BATCH_SIZE)

    print(f"  Train samples : {train_size}")
    print(f"  Val   samples : {val_size}")

    # ── Model ──────────────────────────────────
    print("\n[3/4] Building model …")
    model = NextWordLSTM(
        vocab_size  = config.VOCAB_SIZE,
        embed_dim   = config.EMBED_DIM,
        hidden_dim  = config.HIDDEN_DIM,
        num_layers  = config.NUM_LAYERS,
        dropout     = config.DROPOUT,
    ).to(config.DEVICE)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {total_params:,}")
    print(model)

    # ── Training ───────────────────────────────
    print("\n[4/4] Training …")
    train(model, train_loader, val_loader, config)

    # ── Load best checkpoint for inference ─────
    ckpt  = torch.load(config.MODEL_PATH, map_location=config.DEVICE)
    model.load_state_dict(ckpt["model_state"])

    # ── Interactive Demo ────────────────────────
    print("\n" + "═" * 60)
    print("  NEXT WORD PREDICTION DEMO")
    print("═" * 60)

    demo_seeds = [
        "machine learning is",
        "deep learning uses",
        "the future belongs",
        "knowledge is",
        "once upon a time",
    ]

    for seed in demo_seeds:
        predictions = predict_next_words(
            model, vocab, seed,
            seq_len=config.SEQ_LEN,
            top_k=5,
            temperature=1.0,
            device=config.DEVICE,
        )
        print(f"\n  Seed : \"{seed}\"")
        print(f"  Top-5 predictions:")
        for rank, (word, prob) in enumerate(predictions, 1):
            bar = "█" * int(prob * 40)
            print(f"    {rank}. {word:<15} {prob:.4f}  {bar}")

    print("\n" + "─" * 60)
    print("  TEXT GENERATION (temperature=0.8)")
    print("─" * 60)

    gen_seeds = [
        "machine learning",
        "the future",
        "knowledge is power",
    ]

    for seed in gen_seeds:
        generated = generate_sentence(
            model, vocab, seed,
            seq_len=config.SEQ_LEN,
            num_words=8,
            temperature=0.8,
            device=config.DEVICE,
        )
        print(f"\n  Seed      : \"{seed}\"")
        print(f"  Generated : \"{generated}\"")

    # ── Interactive loop ───────────────────────
    print("\n" + "═" * 60)
    print("  INTERACTIVE MODE  (type 'quit' to exit)")
    print("═" * 60)

    while True:
        user_input = input("\n  Enter seed text: ").strip()
        if user_input.lower() in ("quit", "exit", "q"):
            break
        if not user_input:
            continue

        try:
            temp = float(input("  Temperature (default 1.0): ") or "1.0")
        except ValueError:
            temp = 1.0

        preds = predict_next_words(
            model, vocab, user_input,
            seq_len=config.SEQ_LEN,
            top_k=5,
            temperature=temp,
            device=config.DEVICE,
        )

        print(f"\n  Top-5 next words after \"{user_input}\":")
        for rank, (word, prob) in enumerate(preds, 1):
            bar = "█" * int(prob * 50)
            print(f"    {rank}. {word:<15} {prob:.4f}  {bar}")

        generated = generate_sentence(
            model, vocab, user_input,
            seq_len=config.SEQ_LEN,
            num_words=10,
            temperature=temp,
            device=config.DEVICE,
        )
        print(f"\n  10-word continuation: \"{generated}\"")

    print("\nDone!")


if __name__ == "__main__":
    main()

Device: cuda

[1/4] Preprocessing corpus …
  Vocabulary size : 33
  Total tokens    : 289

[2/4] Building dataset …
  Train samples : 256
  Val   samples : 28

[3/4] Building model …
  Trainable parameters: 934,305
NextWordLSTM(
  (embedding): Embedding(33, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=33, bias=True)
)

[4/4] Training …


C:\Users\dkuns\miniconda3\envs\pytorch_gpu\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(



 Epoch    Train Loss    Val Loss    Perplexity
────────────────────────────────────────────────
     1        3.3402      2.8396         17.11
     2        2.5698      2.4085         11.12
     3        2.3294      2.3514         10.50
     4        2.2045      2.4346         11.41
     5        2.1975      2.3476         10.46
     6        2.1655      2.3028         10.00
     7        2.1461      2.3130         10.10
     8        2.1366      2.3241         10.22
     9        2.1353      2.3399         10.38
    10        2.1383      2.3599         10.59
    11        2.0982      2.3715         10.71
    12        2.0957      2.3789         10.79
    13        2.0878      2.3834         10.84
    14        2.0956      2.3827         10.83
    15        2.0896      2.3813         10.82
    16        2.0723      2.3818         10.82
    17        2.0739      2.3853         10.86
    18        2.0692      2.3865         10.88
    19        2.0356      2.3895         10.91
    20    

C:\Users\dkuns\AppData\Local\Temp\ipykernel_3040\2177407807.py:425: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt  = torch.load(config.MODEL_PATH, map_location=config.D


  Seed : "machine learning is"
  Top-5 predictions:
    1. the             0.0601  ██
    2. is              0.0438  █
    3. a               0.0401  █
    4. in              0.0397  █

  Seed : "deep learning uses"
  Top-5 predictions:
    1. the             0.0899  ███
    2. in              0.0369  █
    3. is              0.0363  █
    4. a               0.0332  █

  Seed : "the future belongs"
  Top-5 predictions:
    1. the             0.0898  ███
    2. in              0.0372  █
    3. is              0.0352  █
    4. a               0.0327  █

  Seed : "knowledge is"
  Top-5 predictions:
    1. the             0.0633  ██
    2. is              0.0432  █
    3. in              0.0408  █
    4. a               0.0406  █

  Seed : "once upon a time"
  Top-5 predictions:
    1. the             0.0910  ███
    2. in              0.0334  █
    3. is              0.0324  █
    4. a               0.0297  █

────────────────────────────────────────────────────────────
  TEXT GENERATION


  Enter seed text:  pytorch
  Temperature (default 1.0):  1.0



  Top-5 next words after "pytorch":
    1. the             0.0754  ███
    2. in              0.0438  ██
    3. is              0.0426  ██
    4. a               0.0401  ██

  10-word continuation: "pytorch <unk> every a <unk> <unk> <unk> the a language is"
